# Intertextuality Evaluator

**The Intertextuality Evaluator** assesses how much a text depends on readers bringing outside knowledge to it for students in grades 3-11. It considers not content knowledge, but relational knowledge: familiarity with other texts, cultural references, genre conventions, and shared discourse that the text assumes without explaining. When you run a passage through the evaluator, it returns a structured output that includes:

* **complexity_score**: The Intertextuality complexity level (Slightly to Exceedingly Complex).
* **adjustment_and_scaffolding**: Analyzing what the author assumes the reader knows vs what is explained.
* **detailed_summary**: Individual complexity factors that drive the rating, with descriptions and their effect on the dimension.
* **recommended_use_cases**: Additional instructional opportunities for using the text.
* **reasoning**: A synthesis of why the text fits the chosen complexity level.

This gives you a clear signal about the intertextuality demands of a passage, helping ensure AI-generated content is appropriate for the target grade.

In [1]:
%pip install -qU langchain-google-genai langchain pydantic textstat typing_extensions

Note: you may need to restart the kernel to use updated packages.


In [ ]:
import getpass
import os
from dotenv import load_dotenv
import json
import hashlib
from pathlib import Path
from typing import List
from enum import Enum
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import JsonOutputParser
from pydantic import BaseModel, Field
import textstat
from IPython.display import Markdown
import pprint as pp

In [3]:
# Check for the API key
load_dotenv()

if "GOOGLE_API_KEY" not in os.environ:
    os.environ["GOOGLE_API_KEY"] = getpass.getpass("Enter your Google AI API key: ")

In [4]:
ASSETS_DIR = Path(".")
CONFIG = json.loads((ASSETS_DIR / "config.json").read_text())

PROMPT_MESSAGES = []
for msg_spec in CONFIG["steps"][0]["prompt"]["messages"]:
    role = msg_spec["role"]
    path = ASSETS_DIR / msg_spec["source_path"]
    text = path.read_text()
    actual = hashlib.sha256(text.encode("utf-8")).hexdigest()
    declared = msg_spec["sha256"]
    assert actual == declared, (
        f"prompt drift detected for role={role!r} ({msg_spec['source_path']}): "
        f"declared {declared[:12]}..., actual on disk {actual[:12]}..."
    )
    PROMPT_MESSAGES.append((role, text))

SYSTEM_PROMPT_TEXT = next(t for r, t in PROMPT_MESSAGES if r == "system")

print(
    f"Loaded {CONFIG['evaluator']['id']} v{CONFIG['evaluator']['version']} "
    f"from {ASSETS_DIR.resolve()}"
)
print(f"  model:       {CONFIG['steps'][0]['model']['name']}")
print(f"  temperature: {CONFIG['steps'][0]['generation']['temperature']}")
for spec, (role, text) in zip(CONFIG["steps"][0]["prompt"]["messages"], PROMPT_MESSAGES):
    sha = hashlib.sha256(text.encode("utf-8")).hexdigest()[:12]
    print(f"    {role:>6}  {spec['source_path']:<14} ({len(text):>5} chars, sha {sha})")

Loaded intertextuality v0_3_light from /Users/achi/Documents/evaluators/evals/literacy/qualitative-text-complexity/intertextuality
  model:       gemini-3-flash-preview
  temperature: 0.0
    system  system.txt     ( 4541 chars, sha 7e48d529b628)
     human  user.txt       (  170 chars, sha 86887ca1bd3c)


In [5]:
class ComplexityLevel(str, Enum):
    SLIGHTLY_COMPLEX = "slightly_complex"
    MODERATELY_COMPLEX = "moderately_complex"
    VERY_COMPLEX = "very_complex"
    EXCEEDINGLY_COMPLEX = "exceedingly_complex"
    MORE_CONTEXT_NEEDED = "more_context_needed"


class DetailedSummaryItem(BaseModel):
    factor: str = Field(description="The specific text complexity factor identified.")
    description: str = Field(description="How this factor manifests in the text.")
    effect_on_complexity_dimension: str = Field(
        description="How this factor affects the reader's ability to understand the text's specific complexity dimension.")


class ScaffoldingItem(BaseModel):
    scaffolding_need: str = Field(description="The complexity factor that requires scaffolding.")
    suggestion: str = Field(description="A specific instructional strategy to support students with this factor.")


class UseCaseItem(BaseModel):
    opportunity: str = Field(description="An instructional opportunity related to the text.")
    suggestion: str = Field(description="A specific way to leverage this text for that instructional purpose.")


class IntertextualityDetails(BaseModel):
    detailed_summary: List[DetailedSummaryItem]
    adjustment_and_scaffolding: List[ScaffoldingItem]
    recommended_use_cases: List[UseCaseItem]


class EvaluatorOutput(BaseModel):
    complexity_score: ComplexityLevel = Field(description="The Intertextuality complexity level for the target grade.")
    reasoning: str = Field(description="A high-level summary of why the text is at this complexity level for the target grade.")
    details: IntertextualityDetails = Field(description="Practical instructional details including scaffolding strategies and recommended use cases.")


# Sanity: schema fields match config
_defs = CONFIG["output_schema"].get("$defs", {})
def _check(name, cls, schema_props):
    py = set(cls.model_fields.keys())
    sc = set(schema_props.keys())
    assert py == sc, f"{name}: pydantic={py} vs config schema={sc}"
_check("EvaluatorOutput", EvaluatorOutput, CONFIG["output_schema"]["properties"])
print("Schema fields match CONFIG['output_schema'].")

Schema fields match CONFIG['output_schema'].


In [6]:
_FK = next(p for p in CONFIG["preprocessing"] if p["id"] == "fk_score")
assert _FK["implementation"]["python"]["library"] == "textstat"

def calculate_fk_score(text) -> float:
    fn = getattr(textstat, _FK["implementation"]["python"]["function"])
    return round(fn(text), 2)

In [7]:
_STEP = CONFIG["steps"][0]


def evaluate_text_complexity(text: str, grade_level: int):
    parser = JsonOutputParser(pydantic_object=EvaluatorOutput)

    fmt_placeholder = _STEP["parser"]["format_instructions_placeholder"]
    prompt_template = ChatPromptTemplate.from_messages(PROMPT_MESSAGES).partial(
        **{fmt_placeholder: parser.get_format_instructions()}
    )

    llm = ChatGoogleGenerativeAI(
        model=_STEP["model"]["name"],
        temperature=_STEP["generation"]["temperature"],
    )

    try:
        fk_score = calculate_fk_score(text)
        print(f"Calculated Flesch-Kincaid Score: {fk_score}")
        inputs = {"text": text, "grade_level": grade_level, "fk_score": fk_score}
        rendered_messages = prompt_template.format_messages(**inputs)
        raw_result = llm.invoke(rendered_messages)
        formatted_result = parser.invoke(raw_result)
        return {
            "rendered_prompt": [m.model_dump() for m in rendered_messages],
            "raw_output": raw_result,
            "raw_text": raw_result.content,
            "formatted_output": formatted_result,
            "usage": getattr(raw_result, "usage_metadata", None),
        }
    except Exception as e:
        return f"Error evaluating text: {e}"

In [8]:
sample_text = """
"Why Is This Program Called Artemis?\nThe first astronauts landed on the Moon in 1969. The missions were called Apollo. The name Apollo came from stories told by Greek people long ago. In the stories, Apollo was a god. Apollo had a twin sister. Her name was Artemis. She was the goddess of the Moon in the Greek stories.\n\nWhat Spacecraft Will Be Used for the Artemis Program?\nNASA has a new rocket. It is the Space Launch System. It is called SLS for short. It is the most powerful rocket in the world. SLS will carry the Orion spacecraft on top. Orion can carry up to four astronauts. Orion will fly around, or orbit, the Moon. The crew will take trips in spacecraft called landers to get to work on the surface of the Moon. When all of their work is finished, the crew will return to Earth aboard Orion.\n\nWhen Will Artemis Go to the Moon?\nThe first Apollo missions were tests. NASA launched the rocket to be sure it was safe for people and work as planned. Artemis will be tested first, too: Artemis 1 launched SLS and Orion with no astronauts on Nov. 16, 2022. Artemis 2 is carrying astronauts. They will circle past the Moon and return to Earth. Artemis 3 will send a crew with the next man to land on the Moon. Artemis 4 will send astronauts to land on the Moon.\n\nWhat Will Artemis Astronauts Do on the Moon?\nThe Artemis 4 crew will visit the Moon’s South Pole. No one has ever been there. At the Moon, astronauts will:\nSearch for the Moon’s water and use it.\nLearn how to live and work on a different planet or moon.\nTest the new tools that astronauts will need for a mission to Mars.\n\nWhy Is the Artemis Program Important?\nThe Moon is a good place to learn new science. NASA will learn more about the Moon, Earth, and even the Sun. The Moon is also a place to learn how astronauts can one day live and work on Mars.\nAstronauts on the Artemis missions will need new tools. Many companies will make these new tools. This will mean new jobs for people and companies on Earth. Other countries will be NASA’s partners for the new Moon missions. They will work on Artemis to bring the world together for a mission to Earth’s nearest neighbor in space."
"""
result = evaluate_text_complexity(text=sample_text, grade_level=3)
pp.pprint(result["formatted_output"] if isinstance(result, dict) else result)

Calculated Flesch-Kincaid Score: 3.3
{'complexity_score': 'slightly_complex',
 'details': {'adjustment_and_scaffolding': [{'scaffolding_need': 'Greek '
                                                                 'Mythology '
                                                                 'Allusions',
                                             'suggestion': 'Provide a visual '
                                                           'aid or a simple '
                                                           'family tree '
                                                           'showing Apollo and '
                                                           'Artemis to '
                                                           'reinforce the '
                                                           "'twin' concept "
                                                           'mentioned in the '
                                                           'text.'},
      

In [9]:
fixtures_path = ASSETS_DIR / CONFIG["fixtures"]["sniff_test_path"]
if not fixtures_path.exists():
    print(f"(no fixtures.json yet at {fixtures_path}; skipping fixture run)")
else:
    fixtures = json.loads(fixtures_path.read_text())
    print(f"Loaded {len(fixtures)} fixtures from {fixtures_path.name}\n")

    _RUBRIC_ORDER = ["slightly_complex", "moderately_complex",
                      "very_complex", "exceedingly_complex"]
    _ALLOW_ADJ = bool(CONFIG["fixtures"]["tolerance"].get("allow_adjacent_levels", False))

    def _score(predicted, expected):
        if predicted == expected:
            return "exact", 0
        if _ALLOW_ADJ and predicted in _RUBRIC_ORDER and expected in _RUBRIC_ORDER:
            d = abs(_RUBRIC_ORDER.index(predicted) - _RUBRIC_ORDER.index(expected))
            if d == 1:
                return "adjacent", d
        return "fail", None

    results = []
    for fx in fixtures:
        expected = fx["expected"]["complexity_level"]
        out = evaluate_text_complexity(text=fx["input"]["text"],
                                       grade_level=fx["input"]["grade_level"])
        if isinstance(out, str):
            results.append({"id": fx["id"], "status": "error",
                             "predicted": None, "expected": expected, "error": out})
            continue
        predicted = out["formatted_output"]["complexity_score"]
        status, _ = _score(predicted, expected)
        results.append({"id": fx["id"], "status": status,
                         "predicted": predicted, "expected": expected,
                         "description": fx.get("description", "")})

    print("=" * 78)
    print(f"{'ID':>5}  {'STATUS':<8}  {'PREDICTED':<22}  {'EXPECTED':<22}  DESCRIPTION")
    print("=" * 78)
    for r in results:
        icon = {"exact": "PASS", "adjacent": "PASS*", "fail": "FAIL", "error": "ERR"}[r["status"]]
        print(f"{r['id']:>5}  {icon:<8}  {(r['predicted'] or '-'):<22}  "
              f"{r['expected']:<22}  {r.get('description','')[:25]}")

    n = len(results)
    n_exact = sum(1 for r in results if r["status"] == "exact")
    n_adj = sum(1 for r in results if r["status"] == "adjacent")
    n_fail = sum(1 for r in results if r["status"] == "fail")
    n_err = sum(1 for r in results if r["status"] == "error")
    print("=" * 78)
    print(f"Summary: {n_exact} exact, {n_adj} adjacent, {n_fail} fail, {n_err} error  "
          f"--  total {n}")
    if _ALLOW_ADJ:
        print("(Adjacency tolerance ON: predictions within +/-1 rubric step count as PASS*.)")

Loaded 4 fixtures from fixtures.json

Calculated Flesch-Kincaid Score: 3.3
Calculated Flesch-Kincaid Score: 4.12
Calculated Flesch-Kincaid Score: 7.33
Calculated Flesch-Kincaid Score: 10.24
   ID  STATUS    PREDICTED               EXPECTED                DESCRIPTION
NASA-001  PASS      slightly_complex        slightly_complex        What Is the Artemis Progr
NASA-074  PASS      moderately_complex      moderately_complex      What Is a Light-Year?
FYM-1106  PASS      moderately_complex      moderately_complex      A Good Night’s Sleep: Nec
CC-3983  PASS*     very_complex            exceedingly_complex     Italy's Violation of Fait
Summary: 3 exact, 1 adjacent, 0 fail, 0 error  --  total 4
(Adjacency tolerance ON: predictions within +/-1 rubric step count as PASS*.)
